In [ ]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")

In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 13
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for even more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 18
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 25
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# these are the final additional 5 epochs after this im DONE

# ==================================================================
# --- NEW CELL: Resuming Training for 5 More Epochs ---
# ==================================================================
print("\n" + "="*80)
print("STARTING PART 2: Resuming training for more more even more more 5 additional epochs...")
print("="*80 + "\n")

# 1. Set the new TOTAL number of epochs
# Original was 8, you want 5 more, so the new total is 13
new_total_epochs = 30
trainer.args.num_train_epochs = new_total_epochs
print(f"Updated total number of epochs to: {new_total_epochs}")

# 2. Call trainer.train() again, resuming from the latest checkpoint
# The trainer will find the latest checkpoint in "ModernBERT_QEvasion_model"
# (from epoch 8) and continue training up to epoch 13.
trainer.train(resume_from_checkpoint=True)

print("Additional 5 epochs of training completed!")


# --- FINAL Evaluation (after {new_total_epochs} total) ---
# This will evaluate the best model found across ALL 13 epochs
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (after {new_total_epochs} total epochs)")
print(f"(Best model from all runs: {trainer.state.best_model_checkpoint})")
print("="*60)

final_test_results = trainer.evaluate()
for key, value in final_test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# --- Optional: Detailed analysis of the NEW best model ---
print("\nDetailed predictions analysis (from new best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Need to re-import these if the session was reset, but typically not needed
# from sklearn.metrics import classification_report, confusion_matrix
# import numpy as np

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                            target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the best model
drive_model_path = "/content/drive/MyDrive/ModernBERT_QEvasion_best_model_focal_newgamma_69"

print("=" * 60)
print("SAVING BEST MODEL TO GOOGLE DRIVE")
print("=" * 60)
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best F1 macro: {trainer.state.best_metric:.4f}")

# Save model and tokenizer
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"\nModel saved to: {drive_model_path}")
print("Done!")

# Optional: list saved files
import os
if os.path.exists(drive_model_path):
    print(f"\nSaved files in {drive_model_path}:")
    for file in os.listdir(drive_model_path):
        file_path = os.path.join(drive_model_path, file)
        size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
        print(f"  - {file} ({size:.1f} MB)")